# Japan BESS Research Lab v2.1 — P1

公开项目布局 × 透明经营贡献情景。A=官方披露/费率；B=模型计算；C=研究假设。公司公开状态不是独立验证交易结果；经营贡献情景不是淘科实际利润或IRR。保护 v1 与 v2 原 Notebook 和研究基线。

从验收包 source/ 根目录顺序执行六个单元。需要 Python NumPy/SciPy 与 Node。完整12,510次 LP 重建使用 `scripts/prepare_p1_flows.py`；本 Notebook 独立复算代表样本，不改写冻结资产。

In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess
import numpy as np
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/lab_core.py').exists())
sys.path.insert(0,str(ROOT/'scripts'))
from lab_core import load_prices, reservation_dispatch, feature_day, fit_ridge, predict_ridge
def read(p):return json.loads((ROOT/p).read_text(encoding='utf-8'))
def sha(p):return hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
lineage=read('research/p1/lineage.json')
for p,h in lineage['protected_files'].items():assert sha(p)==h,p
for p,v in lineage['assets'].items():assert sha(p)==v['sha256'],p
dates,matrices=load_prices();lab=read('dist/data/lab-v2.json')
assert len(dates)==643 and len(dates)*48==30864
assert sha('research/jepx-normalized.csv')==lab['lineage']['source_sha256']
print({'protected_files_unchanged':len(lineage['protected_files']),'p1_assets_verified':len(lineage['assets']),'days':len(dates),'slots':len(dates)*48,'unit':'JPY/kWh','timezone':'Asia/Tokyo'})


{'protected_files_unchanged': 20, 'p1_assets_verified': 15, 'days': 643, 'slots': 30864, 'unit': 'JPY/kWh', 'timezone': 'Asia/Tokyo'}


## 1. 项目自身证据与可识别子集
不继承其他项目的入市状态；未公开为 null。无可信站点坐标，地图采用区域示意。中日容量冲突保留，不用平均数。

In [2]:
projects=read('research/p1/projects.json');rows=projects['projects'];portfolio=projects['portfolio']
assert len(rows)==len({r['project_id'] for r in rows})==portfolio['projects']==9
assert sum(r['rated_power_mw'] for r in rows)==portfolio['public_power_mw']==18
assert abs(sum(r['energy_capacity_mwh'] for r in rows)-72.7)<1e-8
for r in rows:
 assert r['latitude'] is None and r['longitude'] is None
 assert abs(r['duration_hours']-r['energy_capacity_mwh']/r['rated_power_mw'])<1e-9
 assert all(e['project_id']==r['project_id'] for e in r['evidence'])
for field,total in [('jepx_participation_publicly_stated',2),('primary_qualification_publicly_stated',2),('primary_actual_participation_publicly_stated',3)]:
 assert sum(r[field] is True for r in rows)==total
 print(field,total)
second=next(r for r in rows if r['project_id']=='gifu-kakamigahara-2')
assert second['primary_actual_participation_publicly_stated'] is None
assert next(r for r in rows if r['jepx_area']=='chugoku')['grid_connection_date'] is None
assert (ROOT/'research/p1/projects.json').read_bytes()==(ROOT/'dist/data/taoke-projects.json').read_bytes()
print(portfolio);print(projects['parallel_display_notice'])


jepx_participation_publicly_stated 2
primary_qualification_publicly_stated 2
primary_actual_participation_publicly_stated 3
{'projects': 9, 'public_power_mw': 18.0, 'selected_energy_mwh': 72.7, 'energy_disclosure_range_mwh': [72.28, 72.7], 'jepx_participation_stated': 2, 'primary_qualification_stated': 2, 'primary_actual_participation_stated': 3, 'by_area': {'chubu': 3, 'tokyo': 2, 'tohoku': 1, 'kansai': 1, 'kyushu': 1, 'chugoku': 1}, 'population': 'selected uniquely identifiable official disclosed projects; not a complete company portfolio or ownership total'}
公开项目所在地与市场指标的并列展示，不代表淘科当时依据这些指标进行选址，也不推断其内部投资逻辑。


## 2. 费率与成本身份
JEPX 税前按量0.03 JPY/kWh，或会员100万JPY/月。EPRX FY2026税前0.06 JPY/ΔkW·30min；延伸至CY2026 YTD为C。实际会员合同、分摊和项目成本未知。充电能源支出与88%往返效率已在毛套利中，不再重复扣减。

In [3]:
economics=read('research/p1/economics.json');fees=(ROOT/'research/p1/jepx-fees.csv').read_text(encoding='utf-8')
assert '0.03' in fees and '100万円' in fees
for p,h in economics['source_hashes'].items():assert sha(p)==h,p
for key in ['degradation','aggregationPercent','network','otherDaily','activationDaily','flatShare']:
 assert next(x for x in economics['registry'] if x['key']==key)['identity']=='C'
assert all(p['activationDaily'] is None and p['identity']=='C' for p in economics['presets'].values())
print({'presets':economics['presets'],'formulas':economics['formulas']})


{'presets': {'low': {'label': 'Low-cost / 低成本演示', 'identity': 'C', 'degradation': 1, 'aggregationPercent': 0, 'network': 0, 'otherDaily': 0, 'spotFeeMode': 'per_volume', 'spotFee': 0.03, 'flatMonthlyFee': 1000000, 'flatShare': 1, 'activationDaily': None}, 'base': {'label': 'Base / 基准演示', 'identity': 'C', 'degradation': 2, 'aggregationPercent': 5, 'network': 0.5, 'otherDaily': 1000, 'spotFeeMode': 'per_volume', 'spotFee': 0.03, 'flatMonthlyFee': 1000000, 'flatShare': 1, 'activationDaily': None}, 'high': {'label': 'High-cost / 高成本演示', 'identity': 'C', 'degradation': 4, 'aggregationPercent': 10, 'network': 1, 'otherDaily': 3000, 'spotFeeMode': 'per_volume', 'spotFee': 0.03, 'flatMonthlyFee': 1000000, 'flatShare': 1, 'activationDaily': None}}, 'formulas': {'gross': 'spot discharge receipts − charging energy cost + reserved capacity units × scenario price', 'efficiency': '88% round-trip efficiency is already embedded in dispatch/energy balances, not a second money deduction', 'units': 'rese

## 3. 从同一冻结调度重算瀑布和盈亏平衡
成本变化是固定调度扣费情景，未重新优化净贡献。reserve MW 同时决定减少的现货资源和容量单位；机会成本参照同成本、同LP模型的0MW预留。

In [4]:
node_code="""
import {readFileSync} from 'node:fs';
import {operatingContribution} from './dist/p1-economics.js';
const d=JSON.parse(readFileSync('research/p1/economics.json','utf8')),saved=JSON.parse(readFileSync('research/p1/economics-examples.json','utf8'));
const b=d.basis.chubu,cap={reserve:1,price:3,hours:12,ratio:.65,fee:.06},close=(a,b)=>{if(Math.abs(a-b)>1e-5)throw Error(`${a} != ${b}`);};
const out={};
for(const [key,p] of Object.entries(d.presets)){
 const spot=operatingContribution(b.perfect2025,p),combined=operatingContribution(b.frontier[4],p,cap,b.frontier[0]);
 close(spot.net,saved[key].spot2025.net);close(combined.net,saved[key].combined2026.net);
 close(combined.gross-combined.cost,combined.net);close(combined.waterfall.filter(x=>x.key!=='net').reduce((s,x)=>s+x.value,0),combined.net);
 close(operatingContribution(b.frontier[4],{...p,degradation:combined.breakEvenDegradation},cap,b.frontier[0]).net,0);
 close(operatingContribution(b.frontier[4],p,{...cap,price:combined.breakEvenCapacityPrice},b.frontier[0]).opportunityCostAdjusted,0);
 out[key]={spot2025:spot.net,combined2026:combined.net,breakEvenDegradation:combined.breakEvenDegradation,breakEvenCapacityPrice:combined.breakEvenCapacityPrice,activationStatus:combined.activationStatus};
}
const z={...d.presets.low,degradation:0,spotFee:0,spotFeeMode:'custom',aggregationPercent:0,network:0,otherDaily:0};
close(operatingContribution(b.perfect2025,z).net,b.perfect2025.grossRevenue);
const r=operatingContribution(b.frontier[0],{...d.presets.base,activationDaily:999},{...cap,reserve:0},b.frontier[0]);
close(r.capacityGross,0);close(r.capacityCost,0);close(r.opportunityCostAdjusted,0);
console.log(JSON.stringify(out));
"""
scenarios=json.loads(subprocess.check_output(['node','--input-type=module','-e',node_code],cwd=ROOT,text=True,encoding='utf-8'))
assert scenarios['low']['spot2025']>scenarios['base']['spot2025']>scenarios['high']['spot2025']
print(scenarios)


{'low': {'spot2025': 14925145.70304006, 'combined2026': 25618574.3532595, 'breakEvenDegradation': 15.804432008731514, 'breakEvenCapacityPrice': 1.0611207872531008, 'activationStatus': 'Not Modelled / 未建模'}, 'base': {'spot2025': 9488992.01884395, 'combined2026': 21240880.371939432, 'breakEvenDegradation': 14.274655292517, 'breakEvenCapacityPrice': 0.9518610967631912, 'activationStatus': 'Not Modelled / 未建模'}, 'high': {'spot2025': 1020431.912563201, 'combined2026': 14854719.842618741, 'breakEvenDegradation': 12.584228259010194, 'breakEvenCapacityPrice': 0.7572753023747577, 'activationStatus': 'Not Modelled / 未建模'}}


## 4. 独立 LP 复算代表资源状态
五地区 × 0/1/2MW，共15次。原v2连续LP首尾SOC50%，现货功率2−r；静态双向5min余量。容量激活和恢复轨迹依然未建模。

In [5]:
flows=read('research/p1/frontier-flows.json');target='2026-01-19';i=dates.index(target);solves=0
assert flows['lp_solves']==12510 and flows['maximum_daily_gross_difference_jpy']==0
for region,prices in matrices.items():
 for reserve in [0,1,2]:
  actual=reservation_dispatch(prices[i],reserve);solves+=1
  frozen=next(r for r in flows['frontier_flows'][region] if r['reserveMW']==reserve)
  expected=next(d for d in frozen['daily'] if d['date']==target)
  for k in ['grossRevenue','chargingCost','dischargeRevenue','chargeMWh','dischargeMWh','equivalentCycles']:
   assert abs(actual[k]-expected[k])<1e-5,(region,reserve,k)
  assert abs(actual['soc'][-1]-.5)<1e-7
  assert max(actual['chargeMW']+actual['dischargeMW'])<=2-reserve+1e-7
  assert abs(actual['dischargeMWh']-.88*actual['chargeMWh'])<1e-7
print({'independent_lp_solves':solves,'source_total_lp_solves':flows['lp_solves'],'frozen_v2_gross_difference_jpy':0,'charging_and_efficiency_not_deducted_twice':True})


{'independent_lp_solves': 15, 'source_total_lp_solves': 12510, 'frozen_v2_gross_difference_jpy': 0, 'charging_and_efficiency_not_deducted_twice': True}


## 5. P0无未来泄漏不受 P1 影响
再次改动目标日及未来真实价格，核对训练期、特征与岭回归预测相同。预测截断与P1成本模型独立。

仍缺真实CAPEX、固定O&M/土地/保险、融资/税务、聚合合同、偏差罚金、真实容量约定与激活/SOC恢复结算，才能进入完整项目经济性。

In [6]:
target='2026-06-17';i=dates.index(target);cutoff=dates.index('2026-06-01')-1
x=matrices['chubu'];changed=x.copy();changed[i:]=changed[i:]*100+9999
np.testing.assert_array_equal(feature_day(x,dates,i),feature_day(changed,dates,i))
np.testing.assert_array_equal(predict_ridge(fit_ridge(x,dates,cutoff),x,dates,i),predict_ridge(fit_ridge(changed,dates,cutoff),changed,dates,i))
for p,h in lineage['protected_files'].items():assert sha(p)==h,p
print({'future_perturbation_equal':True,'training_end':dates[cutoff],'all_20_original_assets_unchanged':True,'scope':economics['interpretation'],'exclusions':economics['exclusions']})


{'future_perturbation_equal': True, 'training_end': '2026-05-31', 'all_20_original_assets_unchanged': True, 'scope': 'Net Operating Contribution Scenario / 经营贡献情景；不是净利润、IRR、淘科实际项目收益。', 'exclusions': ['实际激活能量/SOC恢复轨迹', '真实偏差结算与罚金', 'CAPEX、融资、固定O&M、土地、保险及项目税务']}
